# biblia-texto-baixar.ipynb — Texto da Bíblia (WEB) pro projeto

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Baixa a **World English Bible** em USFM do ebible.org, converte pra um JSON
único em `pipeline/dados_lexico/web-biblia.json`, e **confere contra o
`40_Matt_02` que já existe** antes de dar por bom.

Com isso, gerar o `roteiro_versiculos.txt` de qualquer capítulo vira uma
chamada de função — acabou a consulta capítulo a capítulo no site.

## Por que USFM (e não o PDF)

O `WEBTEXT.pdf` da página da narração serve pra ler, não pra virar dado: é de
duas colunas, e extrator de texto embaralha as palavras entre elas. O USFM
marca explicitamente parágrafo (`\p`) e poesia (`\q1`) — que é exatamente a
estrutura que o `roteiro_versiculos.txt` do projeto já tem.

## A conferência não é decorativa

O passo 4 compara o Mateus 2 recém-baixado com o arquivo que você já usou pra
gerar vídeo. **Traduções diferentes em inglês batem ~0,83 de similaridade** —
não 0,2, porque compartilham muita palavra. Por isso o limiar é 0,97: frouxo
demais e uma tradução errada passaria despercebida.

Isso importa porque o `alinhar_versiculos()` casa este texto contra a
transcrição do Whisper pra derivar o tempo de cada versículo. Texto de outra
edição degrada o alinhamento **em silêncio** — aparece só no vídeo montado.

---
*WEB: domínio público, sem restrição de copyright (Michael Paul Johnson /
eBible.org).*


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — Drive e módulos                                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

import shutil, sys
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO_MODULOS = Path("/content/pipeline")

if not PASTA_MODULOS.exists():
    raise SystemExit(f"❌ Módulos não encontrados: {PASTA_MODULOS}")

if DESTINO_MODULOS.exists():
    shutil.rmtree(DESTINO_MODULOS)
shutil.copytree(PASTA_MODULOS, DESTINO_MODULOS)
print(f"✅ {len(list(DESTINO_MODULOS.glob('*.py')))} módulos copiados")

# ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
# "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
# traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
# então um copytree logo depois do mount às vezes enxerga só parte dos
# arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
# quebrar muito depois, num import, longe da causa.
#
# A conferência é de três pontas, porque a causa muda o conserto:
#   manifesto  o que o repositório tem  (versionado; chega pela cópia)
#   Drive      o que chegou lá
#   VM         o que a cópia desta célula trouxe
_manifesto = PASTA_MODULOS / "_manifesto.txt"
if not _manifesto.exists():
    print("   ⚠️  sem _manifesto.txt no Drive — ele é versionado no repositório,")
    print("      então rode o repositorio-sincronizar pra trazê-lo")
else:
    _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                  if l.strip() and not l.startswith("#")}
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO_MODULOS.glob("*.py")}

    _fora_do_drive = sorted(_esperados - _no_drive)
    _nao_copiados  = sorted((_esperados & _no_drive) - _na_vm)

    if _nao_copiados:
        # Estão no Drive mas não vieram: é a listagem preguiçosa do mount.
        # Uma segunda passada, com o mount já quente, costuma resolver.
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO_MODULOS / _n)
        _na_vm = {f.name for f in DESTINO_MODULOS.glob("*.py")}
        _nao_copiados = sorted((_esperados & _no_drive) - _na_vm)

    if _fora_do_drive:
        print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
        for _n in _fora_do_drive:
            print(f"     {_n}")
        raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_esperados)} módulos do manifesto estão na VM")

if str(DESTINO_MODULOS) not in sys.path:
    sys.path.insert(0, str(DESTINO_MODULOS))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO — edite só esta célula                          ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. DE ONDE BAIXAR ───────────────────────────────────────────────────────
# O ebible.org publica cada tradução como um zip de USFM. A WEB tem mais de
# uma edição; a que pareia com a narração do David Williams é a **Classic**.
#
# Candidatos tentados EM ORDEM, ficando no primeiro que responder um zip
# válido. Se todos falharem, o notebook para e te manda conferir o link --
# ele não inventa fonte alternativa.
URLS_CANDIDATAS = [
    "https://ebible.org/Scriptures/eng-web-c_usfm.zip",   # WEB Classic
    "https://ebible.org/Scriptures/eng-web_usfm.zip",     # WEB
    "https://ebible.org/Scriptures/engwebp_usfm.zip",     # WEB (protestante)
]

# Se você já sabe o link certo, cole aqui e os candidatos acima são ignorados.
# Pra achar: https://ebible.org/eng-web-c/  ->  seção de downloads.
URL_MANUAL = ""

# ── 1b. CAMINHO À PROVA DE BLOQUEIO ─────────────────────────────────────────
# O ebible.org devolveu 403 pro Colab em 29/ago. Servidor pode bloquear por
# User-Agent (a célula de download já se identifica como navegador agora) e
# também por FAIXA DE IP -- o Colab roda em datacenter do Google, e nesse caso
# nenhum cabeçalho resolve, porque o problema não é como você pede, é de onde.
#
# Contra isso só existe um caminho: baixar no SEU navegador, onde funciona, e
# subir o zip pro Drive. Ponha o nome do arquivo aqui e o download é pulado.
#
#   1. abra https://ebible.org/eng-web-c/ e baixe o zip USFM
#   2. suba pra narrated_video/assets/ no Drive
#   3. escreva o nome do arquivo aqui embaixo
ZIP_NO_DRIVE = ""      # ex: "eng-web-c_usfm.zip"

# ── 2. ONDE SALVAR ──────────────────────────────────────────────────────────
# dados_lexico/ é onde já moram os dados de referência do projeto
# (eventos-biblicos.json, titulos-biblicos.json). O texto bíblico é a mesma
# categoria: imutável, versionado, lido por módulo -- não é planilha.
NOME_SAIDA = "web-biblia.json"

# ── 3. CONFERÊNCIA ──────────────────────────────────────────────────────────
# Capítulo já existente no projeto usado como prova de que o texto baixado é
# mesmo o da narração.
CONFERIR_CONTRA = "40_Matt_02"     # pasta em videos/
CONFERIR_SIGLA, CONFERIR_CAP = "Matt", 2

# Limiar de similaridade. Calibrado: WEB x KJV no mesmo trecho dá ~0,83, então
# qualquer coisa abaixo de 0,97 é suspeita de edição/tradução diferente.
LIMIAR = 0.97

print(f"Saída ......... dados_lexico/{NOME_SAIDA}")
print(f"Conferir ...... {CONFERIR_SIGLA} {CONFERIR_CAP} vs {CONFERIR_CONTRA}")
print(f"Limiar ........ {LIMIAR}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR                                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

import biblia_livros as bl
import biblia_texto as bt

BASE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}")
PASTA_DADOS = BASE / "pipeline" / "dados_lexico"
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
CAMINHO_SAIDA = PASTA_DADOS / NOME_SAIDA

TRABALHO = Path("/content/biblia_usfm"); TRABALHO.mkdir(exist_ok=True)

print(f"📖 {len(bl.LIVROS)} livros, {bl.TOTAL_CAPITULOS} capítulos esperados")
print(f"📁 {CAMINHO_SAIDA}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⬇️  1/4 — OBTER O USFM                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

import urllib.request, urllib.error, zipfile, io

TAMANHO_MINIMO = 1_000_000   # o USFM da Bíblia inteira passa de 1 MB zipado

# Sem User-Agent de navegador, o urllib se anuncia como "Python-urllib/3.x" e
# muito servidor devolve 403 pra isso. Foi o que aconteceu em 29/ago.
CABECALHOS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"),
    "Accept": "application/zip,application/octet-stream,*/*",
}

zip_bytes, url_usada, erros = None, None, []

# ── Caminho 1: o zip que você já baixou à mão e subiu pro Drive ──────────
if ZIP_NO_DRIVE:
    caminho_zip = BASE / "assets" / ZIP_NO_DRIVE
    if not caminho_zip.exists():
        raise SystemExit(f"❌ ZIP_NO_DRIVE aponta pra {caminho_zip}, que não existe.")
    zip_bytes = caminho_zip.read_bytes()
    url_usada = f"(arquivo local) assets/{ZIP_NO_DRIVE}"
    print(f"📦 usando o zip do Drive: {ZIP_NO_DRIVE} ({len(zip_bytes)/1e6:.1f} MB)")

# ── Caminho 2: baixar ────────────────────────────────────────────────────
else:
    for url in ([URL_MANUAL] if URL_MANUAL else URLS_CANDIDATAS):
        print(f"⬇️  tentando {url}")
        try:
            req = urllib.request.Request(url, headers=CABECALHOS)
            with urllib.request.urlopen(req, timeout=120) as r:
                dados = r.read()
        except urllib.error.HTTPError as e:
            # 403 e 404 pedem providências diferentes -- misturar os dois numa
            # mensagem só manda você procurar link novo quando o link está certo.
            pista = {403: "bloqueio (User-Agent ou faixa de IP do Colab)",
                     404: "link mudou ou não existe",
                     429: "pediram demais; espere alguns minutos"}.get(e.code, "")
            print(f"   ✗ HTTP {e.code}" + (f" — {pista}" if pista else ""))
            erros.append((url, e.code))
            continue
        except Exception as e:
            print(f"   ✗ {type(e).__name__}: {e}")
            erros.append((url, type(e).__name__))
            continue

        # Servidor pode responder 200 com página de erro. Só aceita se for zip
        # de verdade e do tamanho esperado -- senão o problema só apareceria na
        # hora de parsear, com uma mensagem que não ajuda.
        if len(dados) < TAMANHO_MINIMO:
            print(f"   ✗ veio {len(dados)/1e3:.0f} KB, esperava > {TAMANHO_MINIMO/1e6:.0f} MB")
            erros.append((url, "pequeno demais")); continue
        if dados[:2] != b"PK":
            print("   ✗ não é um zip")
            erros.append((url, "não é zip")); continue

        zip_bytes, url_usada = dados, url
        print(f"   ✅ {len(dados)/1e6:.1f} MB")
        break

if zip_bytes is None:
    so_403 = all(c == 403 for _, c in erros)
    raise SystemExit(
        "❌ Não consegui o zip USFM.\n\n"
        + ("   Todas as tentativas deram 403. Com User-Agent de navegador já\n"
           "   mandado, isso aponta pra bloqueio por FAIXA DE IP: o ebible.org\n"
           "   recusa o datacenter do Google, onde o Colab roda. Nenhum ajuste\n"
           "   de cabeçalho resolve isso.\n\n" if so_403 else "")
        + "   Caminho garantido — leva 2 minutos:\n"
          "     1. abra https://ebible.org/eng-web-c/ no SEU navegador\n"
          "     2. baixe o zip USFM (funciona: o bloqueio é do datacenter,\n"
          "        não da sua casa)\n"
          "     3. suba o arquivo pra narrated_video/assets/ no Drive\n"
          "     4. escreva o nome dele em ZIP_NO_DRIVE, na Configuração\n"
          "     5. rode esta célula de novo\n\n"
          "   Você só faz isso uma vez na vida do projeto.")

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    z.extractall(TRABALHO)

arquivos = sorted(TRABALHO.rglob("*.usfm")) + sorted(TRABALHO.rglob("*.SFM"))
print(f"📄 {len(arquivos)} arquivos USFM")
if not arquivos:
    raise SystemExit("❌ O zip abriu mas não tem .usfm dentro — é o arquivo certo?")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📖 2/4 — PARSEAR                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

import re

livros_lidos, ignorados, por_alias = {}, [], []

for caminho in arquivos:
    conteudo = caminho.read_text(encoding="utf-8-sig", errors="replace")
    m = re.search(r"^\\id\s+(\w+)", conteudo, re.M)
    codigo = m.group(1).upper() if m else "?"
    try:
        livro, capitulos = bt.parsear_usfm(conteudo)
    except (ValueError, KeyError) as e:
        # Deuterocanônicos e front matter caem aqui -- o zip do ebible.org traz
        # mais coisa que os 66 livros. Guarda o código pra PODER diagnosticar:
        # em 29/ago isto aqui só imprimia "9 arquivos fora do cânone (normal)",
        # e faltavam Ester e Daniel. A informação que resolvia o caso tinha
        # sido resumida embora.
        ignorados.append((caminho.name, codigo, str(e)[:60]))
        continue
    if codigo in bl.ALIASES_USFM:
        por_alias.append((codigo, livro, len(capitulos)))
    livros_lidos[livro.sigla] = capitulos

print(f"✅ {len(livros_lidos)} livros dos 66")

if por_alias:
    print(f"\n🔀 {len(por_alias)} livro(s) vieram na forma com deuterocanônicos:")
    for codigo, livro, n_caps in por_alias:
        extra = n_caps - livro.capitulos
        marca = f"  (+{extra} além do cânone)" if extra > 0 else ""
        print(f"     {codigo} → {livro.nome}: {n_caps} capítulos{marca}")
    print("     Capítulo além do cânone é inofensivo: você nunca pede.")
    print("     ⚠️  Ester grego é o caso a conferir — as adições podem deslocar")
    print("        a numeração de versículo. Antes de fazer vídeo de Ester,")
    print("        confira o capítulo contra a fonte.")

if ignorados:
    print(f"\n⏭️  {len(ignorados)} arquivo(s) fora do cânone de 66:")
    for nome, codigo, motivo in ignorados:
        print(f"     {codigo:5s} {nome}")

faltando = [l.sigla for l in bl.LIVROS if l.sigla not in livros_lidos]
if faltando:
    raise SystemExit(
        f"❌ Livros não encontrados no zip: {faltando}\n\n"
        f"   Olhe a lista de ignorados acima: se o livro que falta está lá com\n"
        f"   um código diferente do esperado, é edição com deuterocanônicos e\n"
        f"   o código precisa entrar em biblia_livros.ALIASES_USFM.")

# Confere a contagem de capítulos livro a livro contra o cânone.
problemas = []
for livro in bl.LIVROS:
    lidos = len(livros_lidos[livro.sigla])
    if lidos != livro.capitulos:
        problemas.append(f"{livro.nome}: {lidos} capítulos, cânone tem {livro.capitulos}")

if problemas:
    print("\n⚠️  divergência na contagem de capítulos:")
    for p in problemas:
        print(f"   {p}")
    print("   (esperado nos livros marcados 🔀 acima; em qualquer outro, investigue)")
else:
    total = sum(len(c) for c in livros_lidos.values())
    print(f"\n✅ {total} capítulos — bate com o cânone ({bl.TOTAL_CAPITULOS})")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 3/4 — CONFERIR CONTRA O CAPÍTULO QUE JÁ EXISTE                ║
# ║  A prova de que este texto é mesmo o da narração                  ║
# ╚══════════════════════════════════════════════════════════════════╝

caminho_existente = (BASE / "videos" / CONFERIR_CONTRA /
                     f"{CONFERIR_CONTRA}_roteiro_versiculos.txt")

conferido = False   # vira True só se a comparação rodar e passar

if not caminho_existente.exists():
    # Pular não pode ser um encolher de ombros: esta comparação é a ÚNICA
    # prova de que o texto baixado é a edição que a narração usa, e o texto
    # errado degrada o alinhamento em silêncio. Se a pasta do vídeo foi
    # renomeada (ex: `40_Matt_02.old` num teste do zero), aponte
    # CONFERIR_CONTRA pra ela -- a prova custa 10 segundos.
    print(f"⚠️  SEM PROVA — {caminho_existente.name} não está em")
    print(f"    {caminho_existente.parent}")
    print()
    print("    O texto vai ser salvo, mas NINGUÉM conferiu que é a mesma")
    print("    edição da narração. Pra provar: aponte CONFERIR_CONTRA pra uma")
    print("    pasta que tenha o roteiro e rode esta célula de novo.")
    pastas = sorted(p.name for p in (BASE / "videos").iterdir()
                    if p.is_dir() and any(p.glob("*_roteiro_versiculos.txt")))
    if pastas:
        print(f"\n    Pastas com roteiro: {', '.join(pastas)}")
else:
    texto_existente = caminho_existente.read_text(encoding="utf-8")
    texto_baixado = bt.gerar_roteiro(livros_lidos[CONFERIR_SIGLA][CONFERIR_CAP])

    c = bt.comparar(texto_existente, texto_baixado)

    print(f"projeto ... {c.palavras_a} palavras")
    print(f"baixado ... {c.palavras_b} palavras")
    print(f"similaridade {c.similaridade:.4f}   (limiar {LIMIAR})")
    print()

    if c.identico:
        print("✅ IDÊNTICOS — o texto do projeto é exatamente esta edição.")
        conferido = True
    elif c.similaridade >= LIMIAR:
        conferido = True
        print(f"✅ MESMA EDIÇÃO, com {len(c.diferencas)} diferença(s) pequena(s):")
        for tipo, a, b in c.diferencas[:10]:
            print(f"   [{tipo}]")
            print(f"     projeto: ...{a}...")
            print(f"     baixado: ...{b}...")
        if len(c.diferencas) > 10:
            print(f"   ... e mais {len(c.diferencas) - 10}")
    else:
        print("❌ EDIÇÃO DIFERENTE.")
        print(f"   Referência: WEB x KJV no mesmo trecho dá ~0,83 — ou seja,")
        print(f"   {c.similaridade:.2f} é compatível com outra tradução, não com ruído.")
        print("   Confira se URLS_CANDIDATAS aponta pra WEB Classic (eng-web-c).")
        for tipo, a, b in c.diferencas[:5]:
            print(f"   [{tipo}]")
            print(f"     projeto: ...{a}...")
            print(f"     baixado: ...{b}...")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💾 4/4 — SALVAR                                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

import json, hashlib
from datetime import datetime, timezone

# `conferido` fica DENTRO do arquivo: daqui a seis meses ninguém vai lembrar
# se a prova rodou, e um JSON que não diz se foi conferido é indistinguível de
# um que foi conferido e passou.
saida = {
    "fonte": url_usada,
    "baixado_em": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    "conferido": conferido,
    "conferido_contra": CONFERIR_CONTRA if conferido else None,
    "livros": {
        sigla: {
            str(cap): [{"n": v.numero, "t": v.texto, "q": v.quebra} for v in versiculos]
            for cap, versiculos in sorted(capitulos.items())
        }
        for sigla, capitulos in livros_lidos.items()
    },
}

texto_json = json.dumps(saida, ensure_ascii=False, indent=1)
CAMINHO_SAIDA.write_text(texto_json, encoding="utf-8")

sha = hashlib.sha256(texto_json.encode()).hexdigest()
print(f"💾 {CAMINHO_SAIDA}")
print(f"   {len(texto_json)/1e6:.1f} MB")
print(f"   sha256 {sha[:16]}...")
print()
print("Este arquivo é IMUTÁVEL — o texto bíblico não muda. Se um dia ele")
print("mudar, foi acidente: o sha256 acima é o que prova.")
print()
if conferido:
    print(f"✅ Conferido contra {CONFERIR_CONTRA} — é a edição da narração.")
else:
    print("⚠️  SALVO SEM PROVA: a conferência da célula 3/4 não rodou.")
    print("   Use, mas confira antes de gerar vídeo de capítulo novo.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔤 SIGLAS DO LIVRO NESTA TRADUÇÃO (opcional)                     ║
# ║  Só rode quando o USFM carregado for coreano ou chinês.          ║
# ╚══════════════════════════════════════════════════════════════════╝
# O indicador de livro:versículo que fica no canto do vídeo mostra a
# abreviação em inglês, coreano e chinês ("Matt/마/太 2:16"). As duas últimas
# saem DAQUI: o USFM traz, no cabeçalho de cada livro, o nome na língua da
# própria tradução em três tamanhos, e o `\toc3` é a abreviação.
#
# Vem daqui e não da memória de ninguém porque a sigla aparece em TODO vídeo
# daquele livro, e uma errada é um erro visível que ninguém revisa duas vezes.
#
# Deixe vazio pra pular.
IDIOMA_DESTA_TRADUCAO = ""      # "ko", "zh", "pt"...

if not IDIOMA_DESTA_TRADUCAO:
    print("⏭️  Pulado (IDIOMA_DESTA_TRADUCAO vazio).")
else:
    import json as _json
    import biblia_texto as _bt

    _achadas, _sem_toc3 = {}, []
    for _caminho in arquivos:
        _codigo, _tocs = _bt.extrair_siglas_usfm(
            _caminho.read_text(encoding="utf-8-sig", errors="replace"))
        try:
            _livro = bl.por_usfm(_codigo)
        except (ValueError, KeyError):
            continue                      # fora dos 66 (deuterocanônico, front matter)
        _abrev = _tocs.get("toc3", "").strip()
        if _abrev:
            _achadas[_livro.sigla] = _abrev
        else:
            _sem_toc3.append(_livro.sigla)

    print(f"🔤 {len(_achadas)} de 66 livros trouxeram \\toc3 em '{IDIOMA_DESTA_TRADUCAO}'")
    if _sem_toc3:
        print(f"   ⚠️  {len(_sem_toc3)} sem \\toc3 (ficam vazios, e o vídeo avisa): "
              f"{', '.join(_sem_toc3[:12])}{'...' if len(_sem_toc3) > 12 else ''}")

    # Mescla com a tabela do repositório, sem apagar o que já está lá.
    _tabela_repo = Path("/content/pipeline/dados_lexico/siglas-livros.json")
    _bruto = _json.loads(_tabela_repo.read_text(encoding="utf-8"))
    _mudou = 0
    for _sigla, _abrev in _achadas.items():
        _linha = _bruto["siglas"].setdefault(_sigla, {})
        if _linha.get(IDIOMA_DESTA_TRADUCAO, "") != _abrev:
            _linha[IDIOMA_DESTA_TRADUCAO] = _abrev
            _mudou += 1
    print(f"   {_mudou} sigla(s) novas ou diferentes das que já estavam na tabela")

    _saida = Path(f"siglas-livros.json")
    _saida.write_text(_json.dumps(_bruto, ensure_ascii=False, indent=1) + "\n",
                      encoding="utf-8")
    print()
    for _s in ("Gen", "Ps", "Matt", "John", "Rev"):
        if _s in _bruto["siglas"]:
            print(f"      {_s:6} {_bruto['siglas'][_s]}")

    from google.colab import files
    files.download(str(_saida))
    print()
    print("⚠️  Este arquivo vai pro REPOSITÓRIO (pipeline/dados_lexico/), não pro")
    print("    Drive: o sincronizador copia dados_lexico por cima, então a cópia")
    print("    do Drive seria sobrescrita na próxima sincronização.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ✅ USAR — gerar o roteiro de qualquer capítulo                    ║
# ╚══════════════════════════════════════════════════════════════════╝

# Daqui pra frente, qualquer capítulo sai numa chamada. Exemplo:
SIGLA, CAPITULO = "John", 3

livro = bl.por_sigla(SIGLA)
roteiro = bt.gerar_roteiro(livros_lidos[SIGLA][CAPITULO])

print(f"{livro.nome_projeto(CAPITULO)}  ({len(roteiro.split())} palavras)")
print("─" * 60)
print(roteiro[:400] + ("..." if len(roteiro) > 400 else ""))
print("─" * 60)
print()
print("Pra salvar na pasta de um vídeo:")
print(f'  destino = BASE / "videos" / "{livro.nome_projeto(CAPITULO)}"')
print(f'  destino.mkdir(parents=True, exist_ok=True)')
print(f'  (destino / f"{livro.nome_projeto(CAPITULO)}_roteiro_versiculos.txt")\\')
print(f'      .write_text(roteiro, encoding="utf-8")')